# Predict ResNet-101 image attributes for MMSD

Load the trained COCO ResNet-101 checkpoint, predict attributes for each MMSD image, remove common English stopwords from the ranked candidates, and save the top five remaining words per image as `extract_all`-style rows. This notebook does not train the model.

This is inference-time filtering: the predictor was trained with stopwords among its classes. For a stronger experiment, exclude stopwords when building the COCO attribute vocabulary and retrain.

In [4]:
import json
from pathlib import Path

import torch
from torch import nn
from torchvision import models, transforms
from PIL import Image
from tqdm.auto import tqdm

CHECKPOINT = Path('/kaggle/input/datasets/fruich/resnet101-coco/resnet101_coco_attributes/resnet101_coco.pt')
MMSD_IMAGES = Path('/kaggle/input/datasets/fruich/mmsd-full/data/raw/dataset_image')
OUTPUT_DIR = Path('/kaggle/working/features/resnet101_coco_attributes_filtered')
BATCH_SIZE = 512
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| GPUs:', torch.cuda.device_count())

Device: cuda | GPUs: 2


In [5]:
checkpoint = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
attributes = checkpoint['attributes']
model = models.resnet101(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(attributes))
state = checkpoint['model']
if next(iter(state)).startswith('module.'):
    state = {key.removeprefix('module.'): value for key, value in state.items()}
model.load_state_dict(state)
model = model.eval().to(DEVICE)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

transform = transforms.Compose([transforms.Resize((448, 448)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
stopwords = set('a an the of with in on and is it are was were be been being to from by for at as into over under after before between through during without within about against among around across up down out off then than that this these those i me my we our you your he she they them his her their its who whom which what where when why how all any both each few more most other some such no nor not only own same so too very can will just should could would do does did have has had here there again further once'.split())
stopword_ids = [i for i, word in enumerate(attributes) if word in stopwords]
assert len(attributes) >= 5 + len(stopword_ids)
print('Excluded stopword classes:', len(stopword_ids))

Excluded stopword classes: 68


In [6]:
image_paths = sorted(MMSD_IMAGES.glob('*.jpg'), key=lambda path: path.stem)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rows = []
with torch.inference_mode():
    for start in tqdm(range(0, len(image_paths), BATCH_SIZE), desc='Predict MMSD attributes'):
        paths = image_paths[start:start + BATCH_SIZE]
        images = []
        for path in paths:
            with Image.open(path) as image:
                images.append(transform(image.convert('RGB')))
        scores = model(torch.stack(images).to(DEVICE)).sigmoid().cpu()
        if stopword_ids:
            scores[:, stopword_ids] = -1
        top_ids = scores.topk(5, dim=1).indices.tolist()
        rows.extend([[path.stem, *[attributes[i] for i in ids]]
                     for path, ids in zip(paths, top_ids)])

output_file = OUTPUT_DIR / 'extract_all'
output_file.write_text(''.join(repr(row) + '\n' for row in rows))
print('Wrote', len(rows), 'top-five attribute rows to', output_file)
print('Examples:', rows[:5])

Predict MMSD attributes:   0%|          | 0/49 [00:00<?, ?it/s]

Wrote 24635 top-five attribute rows to /kaggle/working/features/resnet101_coco_attributes_filtered/extract_all
Examples: [['682716753374351360', 'glass', 'holding', 'wine', 'person', 'drink'], ['682721949072625664', 'picture', 'screen', 'showing', 'sign', 'photo'], ['682722242111774720', 'pictures', 'man', 'photos', 'picture', 'photo'], ['682727470433308672', 'blue', 'picture', 'sign', 'wearing', 'cup'], ['682751920956641280', 'person', 'wearing', 'woman', 'mirror', 'holding']]


In [7]:
import shutil

shutil.make_archive('output_files', 'zip', '/kaggle/working')


'/kaggle/working/output_files.zip'